---
**Author:** Leonardo Gabriel Mourao Thiel  
**Project:** Master Thesis – System Inertia in the Energy System of the Future:
Model-Based Cost Optimization to Secure Inertia Requirements

**Notebook:**  Structural Comparison of Generation Capacities (2024 vs. 2040)


**Date:** 27.04.2026  
---

# Structural Comparison of Generation Capacities (2024 vs. 2040)

This notebook presents a comparative analysis of generation capacities
between the current power system (2024) and a modeled future system (2040).

The objective is to identify structural changes in the generation mix
and assess their implications for system inertia and stability.

---

## Background

The transition towards a low-carbon energy system is expected to
significantly alter the composition of generation capacities.

Key trends include:

- a reduction in conventional thermal generation  
- a strong expansion of renewable energy sources  
- increasing deployment of storage technologies  

These changes directly influence the availability of system inertia,
as different technologies contribute differently to frequency stability.

---

## Objective

The objective of this notebook is to:

- compare installed capacities across technologies and countries  
- quantify structural changes between 2024 and 2040  
- analyze shifts from conventional to renewable generation  
- provide a structural explanation for changes in system inertia  

---

## Methodological Approach

The analysis follows these main steps:

1. Load and process capacity data for 2024  
2. Construct a detailed capacity dataset for 2040  
3. Harmonize technology categories across both datasets  
4. Align data structures for comparability  
5. Compute aggregated capacities and system totals  
6. Export results for further analysis and visualization  

---

## Key Concepts

- **Generation capacity (MW)** as structural indicator  
- **Technology classification** to ensure comparability  
- **System-wide totals** for aggregate analysis  
- **Technology shares** to assess structural shifts  

---

## Scope

- Comparison between:
  - historical system (2024)  
  - modeled future system (2040)  
- Spatial scope: multi-country European system  
- Aggregation by generation technology  

---

## Output

The notebook produces:

- harmonized capacity datasets for 2024 and 2040  
- system-wide capacity comparisons  
- structured datasets for visualization and interpretation  

These results provide the structural basis for explaining differences
in system inertia observed in subsequent analyses.

---

## Interpretation Focus

The analysis aims to answer:

- How does the generation mix change between 2024 and 2040?  
- Which technologies expand or decline?  
- How do these changes affect system inertia availability?  

---

## Notes

This notebook focuses on structural system analysis.

The computation of system inertia and scenario modeling is performed
in separate notebooks.

All results are prepared for reproducibility and integration into
the thesis.

## 1. Environment Setup and Analysis Configuration

This section initializes the analysis environment and defines the
core configuration for the capacity comparison between 2024 and 2040.

It includes:

- loading required Python packages  
- installing dependencies for reproducibility  
- defining input and output paths  
- specifying the set of countries included in the analysis  

These settings provide the foundation for loading and processing
capacity data in the subsequent steps.

In [1]:

import sys

# Install dependencies (reproducibility)
!{sys.executable} -m pip install -r requirements.txt

# Custom plotting functions
import plots_2024 as p
import pandas as pd
import numpy as np
import os

# ---------------------------------------------------------
# Path configuration
# ---------------------------------------------------------

path = os.path.join("..", "Data")
output_path = os.path.join("..", "Results")


# ---------------------------------------------------------
# Countries included in the analysis
# ---------------------------------------------------------

countryList = [
    "AT","BA","BE","BG","CH","CZ","DE","DK","ES","FR","GR"
]

countryList += [
    "HR","HU","IT","LU","MK","ME","NL","PL","PT","RO","RS","SI","SK"
]

ERROR: Could not find a version that satisfies the requirement os (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: C:\Users\Leo\AppData\Local\Python\pythoncore-3.12-64\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for os


## Processing of Thermal Generation Capacities (2040)

This section processes thermal generation capacity data and aggregates
it by country and fuel type.

Thermal power plants are particularly relevant for system inertia,
as they typically provide the majority of rotational inertia in
the power system.

---

### Purpose

- disaggregate thermal capacity by fuel type  
- enable technology-specific analysis  
- support the interpretation of inertia contributions  

---

### Method

- load thermal capacity dataset  
- clean and standardize variables  
- filter for relevant countries  
- aggregate capacities by country and fuel type  
- reshape data into a pivoted format  

In [2]:
# ---------------------------------------------------------
# Load thermal capacity data (2040)
# ---------------------------------------------------------

file_path = os.path.join(
    path,
    "2040",
    "capacities_2040",
    "thermal_cleaned.csv"
)

df = pd.read_csv(file_path, sep=";", encoding="latin1")


# ---------------------------------------------------------
# Data cleaning
# ---------------------------------------------------------

# Standardize column names
df.columns = df.columns.str.strip().str.lower()

# Ensure correct data types
df["fuel_type"] = df["fuel_type"].astype(str).fillna("Unknown")
df["country"] = df["country"].astype(str)

df["number_of_units"] = df["number_of_units"].fillna(1).astype(int)
df["net_capacity_mw"] = df["net_capacity_mw"].fillna(0).astype(float)


# ---------------------------------------------------------
# Filter for selected countries
# ---------------------------------------------------------

df = df[df["country"].isin(countryList)]


# ---------------------------------------------------------
# Compute total capacity
# ---------------------------------------------------------

# (Currently identical to net capacity, but kept explicit for clarity)
df["total_capacity_mw"] = df["net_capacity_mw"]


# ---------------------------------------------------------
# Aggregate by country and fuel type
# ---------------------------------------------------------

result = (
    df
    .groupby(["country", "fuel_type"], as_index=False)["total_capacity_mw"]
    .sum()
    .sort_values(["country", "total_capacity_mw"], ascending=[True, False])
)


# ---------------------------------------------------------
# Pivot to wide format
# ---------------------------------------------------------

df_pivot = result.pivot_table(
    index="country",
    columns="fuel_type",
    values="total_capacity_mw",
    aggfunc="sum",
    fill_value=0
)



## Integration of Non-Renewable Capacities (2040)

In this step, non-renewable generation capacities are added to the
existing capacity dataset for 2040.

The data is aggregated at the country level and merged with the
previously prepared dataset.

---

### Purpose

- complete the capacity representation across technologies  
- ensure consistency between renewable and non-renewable capacities  
- enable a full comparison of generation structure  

---

### Method

- load non-renewable capacity data  
- aggregate capacities per country  
- merge with the main dataset  

In [3]:
# ---------------------------------------------------------
# Load non-renewable capacity data (2040)
# ---------------------------------------------------------

file_path = os.path.join(
    path,
    "2040",
    "capacities_2040",
    "other_nonres_capacity.csv"
)

df_cap = pd.read_csv(file_path, sep=";", low_memory=False)


# ---------------------------------------------------------
# Aggregate capacities per country
# ---------------------------------------------------------

df_country = (
    df_cap
    .groupby("country", as_index=False)["capacity_MW"]
    .sum()
)


# ---------------------------------------------------------
# Merge with existing dataset
# ---------------------------------------------------------

# df_pivot should already contain other capacity categories
if "df_pivot" not in locals():
    raise ValueError("df_pivot is not defined before merging")

df_pivot = df_pivot.merge(df_country, on="country", how="left")


# Rename column for clarity
df_pivot = df_pivot.rename(columns={"capacity_MW": "nonres"})




## Integration of Renewable Capacities (2040)

In this step, renewable generation capacities are processed and
integrated into the existing capacity dataset.

The data is aggregated by country and technology, allowing for a
detailed representation of the renewable generation mix.

---

### Purpose

- include renewable technologies in the capacity dataset  
- enable technology-level comparison  
- prepare a complete dataset for structural analysis  

---

### Method

- load renewable capacity data  
- filter for the target year (2040)  
- clean and convert numerical values  
- aggregate capacities by country and technology  
- reshape data into wide format (pivot)  
- merge with existing dataset  

In [4]:
# ---------------------------------------------------------
# Load renewable capacity data (2040)
# ---------------------------------------------------------

file_path = os.path.join(
    path,
    "2040",
    "capacities_2040",
    "other_res_capacity.csv"
)

df_other = pd.read_csv(
    file_path,
    sep=";",
    encoding="latin1"
)


# ---------------------------------------------------------
# Filter for target year
# ---------------------------------------------------------

df_other = df_other[df_other["year"] == 2040]


# ---------------------------------------------------------
# Data cleaning
# ---------------------------------------------------------

# Convert installed capacity to numeric (invalid → NaN → 0)
df_other["installed_capacity_MW"] = pd.to_numeric(
    df_other["installed_capacity_MW"],
    errors="coerce"
).fillna(0)


# ---------------------------------------------------------
# Aggregate by country and technology
# ---------------------------------------------------------

df_other_agg = (
    df_other
    .groupby(["country", "technology"], as_index=False)["installed_capacity_MW"]
    .sum()
)


# ---------------------------------------------------------
# Pivot to wide format (technologies as columns)
# ---------------------------------------------------------

df_other_pivot = df_other_agg.pivot_table(
    index="country",
    columns="technology",
    values="installed_capacity_MW",
    aggfunc="sum",
    fill_value=0
)


# ---------------------------------------------------------
# Prepare for merge
# ---------------------------------------------------------

df_pivot = df_pivot.reset_index()
df_other_pivot = df_other_pivot.reset_index()


# ---------------------------------------------------------
# Merge datasets
# ---------------------------------------------------------

df_final = (
    df_pivot
    .merge(df_other_pivot, on="country", how="left")
    .fillna(0)
)




## Integration of Renewable Generation Capacities (Solar and Wind)

This section integrates renewable generation capacities into the
final dataset, focusing on solar and wind technologies.

Solar capacity is aggregated from rooftop and utility-scale PV,
while wind capacity is separated into onshore and offshore components.

---

### Purpose

- complete the representation of renewable generation technologies  
- distinguish between different renewable sources  
- enable detailed comparison of generation structure  

---

### Method

- load capacity data from renewable datasets  
- aggregate solar capacity (rooftop + PV)  
- structure wind capacity (onshore and offshore)  
- merge all components into the final dataset  

In [5]:
from renewable_generation import get_capacity

# ---------------------------------------------------------
# Load renewable capacities
# ---------------------------------------------------------

cap_roof, cap_pv, cap_on, cap_off = get_capacity(path)


# ---------------------------------------------------------
# Aggregate solar capacity (rooftop + utility-scale PV)
# ---------------------------------------------------------

solar_capacity = {}

all_countries = set(cap_roof) | set(cap_pv)

for c in all_countries:
    solar_capacity[c] = cap_roof.get(c, 0) + cap_pv.get(c, 0)


# Convert to DataFrames
df_solar = pd.DataFrame(
    list(solar_capacity.items()),
    columns=["country", "solar"]
)

df_onshore = pd.DataFrame(
    list(cap_on.items()),
    columns=["country", "wind_onshore"]
)

df_offshore = pd.DataFrame(
    list(cap_off.items()),
    columns=["country", "wind_offshore"]
)


# ---------------------------------------------------------
# Merge into final dataset
# ---------------------------------------------------------

df_final = df_final.merge(df_solar, on="country", how="left")
df_final = df_final.merge(df_onshore, on="country", how="left")
df_final = df_final.merge(df_offshore, on="country", how="left")


# ---------------------------------------------------------
# Handle missing values
# ---------------------------------------------------------

df_final = df_final.fillna(0)




## Integration of Hydropower Capacities

This section incorporates hydropower capacities into the final dataset.

Hydropower plays a unique role in the power system, as it can contribute
to system inertia depending on the turbine type and operational mode.

---

### Purpose

- complete the generation capacity dataset  
- include hydropower as a distinct technology  
- enable a more accurate representation of inertia-relevant resources  

---

### Method

- load hydropower capacity data  
- transform dataset into country-based structure  
- merge with the existing dataset  

In [6]:
from hydro_data import load_hydro_capacities

# ---------------------------------------------------------
# Load hydropower capacities
# ---------------------------------------------------------

hydro_df = load_hydro_capacities(path, countryList)


# ---------------------------------------------------------
# Transform structure (countries as rows)
# ---------------------------------------------------------

hydro_df = (
    hydro_df.T
    .reset_index()
    .rename(columns={"index": "country"})
)


# ---------------------------------------------------------
# Merge into final dataset
# ---------------------------------------------------------

df_final = (
    df_final
    .merge(hydro_df, on="country", how="left")
)


# ---------------------------------------------------------
# Handle missing values
# ---------------------------------------------------------

df_final = df_final.fillna(0)



C:\Users\Leo\AppData\Local\Temp\ipykernel_9968\3085075620.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_final = df_final.fillna(0)


## Integration of Battery Storage Capacities

This section incorporates battery storage capacities into the final dataset.

Battery systems do not provide inherent rotational inertia, but they can
contribute to system stability through fast frequency response and
synthetic (virtual) inertia.

---

### Purpose

- include storage technologies in the capacity dataset  
- capture flexibility resources in the system  
- support the analysis of alternative stability mechanisms  

---

### Method

- load battery capacity data  
- filter for the target year (2040)  
- aggregate capacities at the country level  
- merge with the final dataset  

In [7]:
# ---------------------------------------------------------
# Load battery capacity data (2040)
# ---------------------------------------------------------

file_path = os.path.join(
    path,
    "2040",
    "capacities_2040",
    "battery_cleaned.csv"
)

df_bat = pd.read_csv(file_path, sep=";", encoding="latin1")


# ---------------------------------------------------------
# Data cleaning
# ---------------------------------------------------------

df_bat["country"] = df_bat["country"].astype(str)

df_bat["net_maximum_capacity_generation_perspective"] = (
    df_bat["net_maximum_capacity_generation_perspective"]
    .fillna(0)
    .astype(float)
)


# ---------------------------------------------------------
# Filter for target year
# ---------------------------------------------------------

df_bat = df_bat[df_bat["year"] == 2040]


# ---------------------------------------------------------
# Aggregate by country
# ---------------------------------------------------------

df_bat_country = (
    df_bat
    .groupby("country", as_index=False)["net_maximum_capacity_generation_perspective"]
    .sum()
    .rename(columns={
        "net_maximum_capacity_generation_perspective": "battery_MW"
    })
)


# ---------------------------------------------------------
# Merge into final dataset
# ---------------------------------------------------------

df_final = (
    df_final
    .merge(df_bat_country, on="country", how="left")
    .fillna(0)
)



## Loading Generation Capacities for 2024

This section loads generation capacity data for the reference year 2024.

The data is stored on a country-by-country basis and serves as the
baseline for comparison with the modeled 2040 system.

---

### Purpose

- provide a reference system for comparison  
- enable analysis of structural changes in generation capacity  
- support interpretation of differences in system inertia  

---

### Method

- load country-specific capacity datasets  
- store data in a structured dictionary  
- prepare for further aggregation and comparison  

In [8]:
# ---------------------------------------------------------
# Load capacity data for 2024
# ---------------------------------------------------------

capacity_2024 = {}

for country in countryList:

    # Build file path
    filepath = os.path.join(path, "capacity", f"{country}.xlsx")

    # Check if file exists
    if not os.path.exists(filepath):
        print(f" {country} does not exist")
        continue

    # Load Excel file
    df = pd.read_excel(
        filepath,
        header=0,
        skiprows=5,
        engine="openpyxl"
    )

    # Store in dictionary
    capacity_2024[country] = df



## Aggregation and Restructuring of Capacity Data (2024)

In this step, the country-level capacity datasets for 2024 are combined
and transformed into a unified structure.

The data is aggregated by country and technology and reshaped into a
pivoted format to enable direct comparison with the 2040 dataset.

---

### Purpose

- unify country-level datasets into a single DataFrame  
- clean and standardize capacity values  
- aggregate capacities by technology  
- create a comparable structure for 2024 vs. 2040  

---

### Method

- concatenate country datasets  
- clean and convert capacity values  
- remove non-relevant entries  
- pivot data into wide format (technologies as columns)  

In [9]:
# ---------------------------------------------------------
# Combine all country datasets into one DataFrame
# ---------------------------------------------------------

df_list = []

for country, df_2024 in capacity_2024.items():
    temp = df_2024.copy()
    temp["country"] = country
    df_list.append(temp)

df_2024 = pd.concat(df_list, ignore_index=True)


# ---------------------------------------------------------
# Standardize column names
# ---------------------------------------------------------

df_2024.columns = ["technology", "capacity_MW", "country"]


# ---------------------------------------------------------
# Data cleaning
# ---------------------------------------------------------

df_2024["capacity_MW"] = (
    df_2024["capacity_MW"]
    .replace("n/e", 0)
    .fillna(0)
    .astype(float)
)

# Remove total rows (avoid double counting)
df_2024 = df_2024[df_2024["technology"] != "Total Grand Capacity"]


# ---------------------------------------------------------
# Pivot to country × technology matrix
# ---------------------------------------------------------

df_2024_pivot = df_2024.pivot_table(
    index="country",
    columns="technology",
    values="capacity_MW",
    aggfunc="sum",
    fill_value=0
)

# Remove column index name for cleaner output
df_2024_pivot.columns.name = None

# Reset index for merging
df_2024_pivot = df_2024_pivot.reset_index()




## Harmonization of Technology Categories (2024 vs. 2040)

In this step, generation technologies from the 2040 dataset are mapped
to the corresponding categories used in the 2024 dataset.

Due to differences in data sources and modeling approaches, technology
definitions are not directly comparable and must be aligned.

---

### Purpose

- ensure consistency between datasets  
- enable direct comparison of capacity structures  
- avoid mismatches in technology definitions  

---

### Method

- define mapping rules between 2040 and 2024 technologies  
- apply direct mappings where categories match exactly  
- aggregate multiple 2040 technologies into unified 2024 categories  

---

### Importance

This harmonization step is essential for ensuring that observed
differences between 2024 and 2040 reflect actual system changes,
rather than inconsistencies in data structure.

In [10]:
mapping_2040_to_2024 = {
    # Fossil
    "Gas": "Fossil Gas",
    "Hard coal": "Fossil Hard coal",
    "Lignite": "Fossil Brown coal/Lignite",
    "Heavy oil": "Fossil Oil",
    "Light oil": "Fossil Oil",
    "Oil shale": "Fossil Oil shale",

    # Nuclear
    "Nuclear": "Nuclear",

    # Renewables
    "geothermal": "Geothermal",
    "marine": "Marine",

    # Biomass / Waste
    "small_biomass": "Biomass",
    "waste": "Waste",

    # Storage
    "battery_MW": "Energy storage",

    # Other
    "Hydrogen": "Other",
    "nonres": "Other",
    "not_defined_/_splitting_not_known": "Other",
}
direct_map = {
    # Solar / Wind
    "solar": "Solar",
    "wind_onshore": "Wind Onshore",
    "wind_offshore": "Wind Offshore",

    # Hydro (jetzt sauber!)
    "Run of River - MW": "Hydro Run-of-river and pondage",
    "Pondage - MW": "Hydro Run-of-river and pondage",

    "Reservoir - MW": "Hydro Water Reservoir",

    "PS Open (turbine) - MW": "Hydro Pumped Storage",
    "PS Closed (turbine) - MW": "Hydro Pumped Storage",
}
# ---------------------------------------------------------
# Create working copy
# ---------------------------------------------------------

df_2040 = df_final.copy()

# Remove unnecessary columns if present
df_2040 = df_2040.drop(columns=["index"], errors="ignore")


# ---------------------------------------------------------
# Initialize mapped DataFrame
# ---------------------------------------------------------

df_2040_mapped = pd.DataFrame()
df_2040_mapped["country"] = df_2040["country"]


# ---------------------------------------------------------
# Apply mapping
# ---------------------------------------------------------

for col in df_2040.columns:

    if col == "country":
        continue

    # Priority 1: direct mapping (exact match)
    if col in direct_map:
        new_col = direct_map[col]

    # Priority 2: general mapping
    else:
        new_col = mapping_2040_to_2024.get(col)

    # Skip unmapped technologies
    if new_col is None:
        continue

    # Initialize column if not present
    if new_col not in df_2040_mapped.columns:
        df_2040_mapped[new_col] = 0

    # Aggregate values
    df_2040_mapped[new_col] += df_2040[col]



## Alignment and Aggregation of Capacity Data (2024 vs. 2040)

In this step, the 2024 and 2040 capacity datasets are aligned to ensure
full comparability across countries and technologies.

Missing technologies are added where necessary, and both datasets are
extended with total capacity values.

---

### Purpose

- ensure identical structure across both datasets  
- enable consistent comparison across technologies  
- compute total system capacity  
- include aggregated system-level values  

---

### Method

- add missing technology columns to 2040 dataset  
- reorder columns to match 2024 structure  
- compute total capacity per country  
- add aggregated "TOTAL" row for system-wide comparison  

In [11]:
# ---------------------------------------------------------
# Ensure identical technology columns
# ---------------------------------------------------------

for col in df_2024_pivot.columns:
    if col not in df_2040_mapped.columns:
        df_2040_mapped[col] = 0

# Align column order
df_2040_mapped = df_2040_mapped[df_2024_pivot.columns]


# ---------------------------------------------------------
# Compute total capacity per country
# ---------------------------------------------------------

df_2040_mapped["Total_Capacity_MW"] = (
    df_2040_mapped
    .drop(columns=["country"])
    .sum(axis=1)
)

df_2024_pivot["Total_Capacity_MW"] = (
    df_2024_pivot
    .drop(columns=["country"])
    .sum(axis=1)
)


# ---------------------------------------------------------
# Compute system-wide totals (all countries)
# ---------------------------------------------------------

# 2024
total_row_2024 = df_2024_pivot.drop(columns=["country"]).sum()
total_row_2024["country"] = "TOTAL"

df_2024_with_total = pd.concat(
    [df_2024_pivot, pd.DataFrame([total_row_2024])],
    ignore_index=True
)

# 2040
total_row_2040 = df_2040_mapped.drop(columns=["country"]).sum()
total_row_2040["country"] = "TOTAL"

df_2040_with_total = pd.concat(
    [df_2040_mapped, pd.DataFrame([total_row_2040])],
    ignore_index=True
)

## Export of Capacity Comparison Results

This section exports the aligned and aggregated capacity datasets
for 2024 and 2040 to an Excel file.

The exported data includes both country-level values and system-wide
totals, enabling a direct comparison of generation structures.

---

### Purpose

- provide a structured dataset for further analysis  
- support visualization and reporting  
- ensure reproducibility of results  

---

### Output

The Excel file contains:

- **Capacity_2024**: reference system capacities  
- **Capacity_2040**: future scenario capacities  

In [12]:
# ---------------------------------------------------------
# Define output path
# ---------------------------------------------------------

output_file = os.path.join(output_path, "capacity_comparison.xlsx")


# ---------------------------------------------------------
# Optional: compute difference (2040 - 2024)
# ---------------------------------------------------------

df_diff = df_2040_with_total.copy()
df_diff.set_index("country", inplace=True)

df_2024_temp = df_2024_with_total.set_index("country")

# Align indices just in case
df_diff = df_diff - df_2024_temp
df_diff.reset_index(inplace=True)


# ---------------------------------------------------------
# Export to Excel
# ---------------------------------------------------------

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    df_2024_with_total.to_excel(writer, sheet_name="Capacity_2024", index=False)

    df_2040_with_total.to_excel(writer, sheet_name="Capacity_2040", index=False)

    # 🔥 sehr wertvoll für Analyse
    df_diff.to_excel(writer, sheet_name="Difference_2040-2024", index=False)


print("Done:", output_file)

Done: ..\Results\capacity_comparison.xlsx
